# 階段 4：C 組回測 —— 只有第一層濾網

台股擇時策略研究專案第四階段。**這是第一次產出績效數字。**

```
第一層（階段 3 已完成）：MA50 月度濾網  →  決定「可不可以做多」
第二層（階段 5、6）    ：RSI 回檔進場     →  決定「什麼時候進」
```

本階段的 **C 組 = 只有第一層**：狀態轉為可做多就進場，狀態轉為空手就出場。

### C 組的角色是基準線

階段 6 的 B 組（加上 RSI 擇時）要跟 C 組比，才能回答「RSI 這一層到底加了什麼」。
先有 C 組的數字，B 組出來時貢獻是正是負才一目了然。

**本階段完全不使用 RSI，也不做 B 組。**

### 沿用階段 1–3 的結論（已確認，不重複檢查）

- `data/regime.csv`，6,632 筆，1999-01-05 ~ 2026-01-20
- `regime` 為 `LONG_OK` / `FLAT`，2000-01-01 之後無缺值
- 狀態切換已驗證無 look-ahead：月底判斷，次月第一個交易日生效
- 研究期間 48 次狀態切換，其中 24 次 `FLAT → LONG_OK`；`LONG_OK` 佔 66.72%

**產出**：`data/backtest_C.csv`、`data/trades_C.csv`

---
## 研究設定（本專案共通）

| 項目 | 設定 |
|---|---|
| 研究期間 | 2000-01-01 ~ 2026-01-20 |
| IS | 2000-01-01 ~ 2017-12-31 |
| OOS | 2018-01-01 ~ 2026-01-20 |
| 曝險 | 在場 100%，空手 0，無槓桿 |
| 報酬計算 | 每日報酬序列累乘（複利） |
| 空手期間 | 報酬為 0（不給利息） |
| 無風險利率 | 0（與空手不給利息保持內部一致） |
| 交易成本 | **本階段不計** |
| 標的 | 加權指數，**不含股息** |

## 規則定義

### C 組進出場

```
regime 由 FLAT 轉為 LONG_OK 的那一天  →  該日開盤買進
regime 由 LONG_OK 轉為 FLAT 的那一天  →  該日開盤賣出
```

**`regime` 欄位本身已經是「生效後」的狀態序列**（階段 3 已處理過 shift）。
所以 `regime_change == True` 的那一天，就是新狀態生效的第一天，
直接用**該日開盤價**成交，**不再往後 shift 一天**。

重複 shift 會讓進場延後一天，是這類回測最常見的錯誤之一，
區塊 2 會明確驗證這件事。

### 每日報酬

| 情境 | 公式 |
|---|---|
| 進場日（開盤買進，持有到收盤） | `Close[t] / Open[t] − 1` |
| 持有中 | `Close[t] / Close[t−1] − 1` |
| 出場日（持有到開盤賣出） | `Open[t] / Close[t−1] − 1` |
| 空手 | `0` |

切換日的開盤跳空有時不小，所以進出場日不能簡單套用收盤對收盤。

### A 組（買進持有）

研究期間第一個交易日**開盤**買進，持有到最後一個交易日**收盤**。
第一日報酬 `Close / Open − 1`，之後 `Close[t] / Close[t−1] − 1`。

### 研究期間起點的處理

研究期間第一個交易日（2000-01-04）的 `regime` 已是 `LONG_OK`
（該狀態由 1999-12 月底判斷產生，生效日早於研究期間起點）。
C 組與 A 組**都在這一天開盤進場**，起跑點相同，比較才公平。

---
## 1. 參數與載入

參數集中定義。載入後比對筆數、起訖日期與欄位是否與階段 3 一致，不一致就 `raise`。

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============ 參數 ============
DATA_IN     = "data/regime.csv"
DATA_OUT    = "data/backtest_C.csv"
TRADES_OUT  = "data/trades_C.csv"
STUDY_START = "2000-01-01"
IS_END      = "2017-12-31"
OOS_START   = "2018-01-01"
TRADING_DAYS_PER_YEAR = 245

LONG_OK, FLAT = "LONG_OK", "FLAT"
RISK_FREE = 0.0          # 與「空手不給利息」保持內部一致

# ============ 階段 3 已確認的事實 ============
EXPECTED_ROWS  = 6632
EXPECTED_FIRST = "1999-01-05"
EXPECTED_LAST  = "2026-01-20"
EXPECTED_COLS  = ["Open", "High", "Low", "Close", "Volume", "MA",
                  "RSI3", "RSI5", "regime", "is_month_end", "regime_change"]
EXPECTED_LONG_OK_DAYS = 3908      # 研究期間 regime == LONG_OK 的天數

pd.set_option("display.width", 170)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14, 7)

# ============ 驗證結果收集器（沿用階段 2、3 模式） ============
CHECKS = []

def record(name, passed, detail=""):
    verdict = "通過" if passed else "異常"
    CHECKS.append({"驗證項目": name, "結果": verdict, "說明": detail})
    print(f"===> [{verdict}] {name}" + (f"\n      {detail}" if detail else ""))

print("參數設定完成")
print(f"  研究期間 : {STUDY_START} ~ {EXPECTED_LAST}")
print(f"  IS / OOS : ~{IS_END} / {OOS_START}~")
print(f"  年化基準 : {TRADING_DAYS_PER_YEAR} 交易日")
print(f"  交易成本 : 不計（本階段設定）")

參數設定完成
  研究期間 : 2000-01-01 ~ 2026-01-20
  IS / OOS : ~2017-12-31 / 2018-01-01~
  年化基準 : 245 交易日
  交易成本 : 不計（本階段設定）


In [2]:
raw = pd.read_csv(DATA_IN, index_col="Date", parse_dates=True).sort_index()

load_check = pd.DataFrame({
    "階段 3 確認值": [str(EXPECTED_ROWS), EXPECTED_FIRST, EXPECTED_LAST,
                      ", ".join(EXPECTED_COLS)],
    "本次載入值":    [f"{len(raw)}", f"{raw.index.min():%Y-%m-%d}",
                      f"{raw.index.max():%Y-%m-%d}", ", ".join(raw.columns)],
}, index=["筆數", "起始日期", "結束日期", "欄位"])
load_check["一致"] = np.where(
    load_check["階段 3 確認值"] == load_check["本次載入值"], "是", "★ 否")
display(load_check)

if (load_check["一致"] == "★ 否").any():
    raise RuntimeError("載入的資料與階段 3 不一致，請停止並人工確認 data/regime.csv")

df = raw.loc[STUDY_START:].copy()          # 研究期間
prev_close_first = raw["Close"].shift(1).loc[df.index[0]]   # 研究期間首日的前一日收盤

record("資料載入一致性", True,
       f"{len(raw):,} 筆與階段 3 完全一致；研究期間切出 {len(df):,} 個交易日"
       f"（{df.index.min():%Y-%m-%d} ~ {df.index.max():%Y-%m-%d}）")

,階段 3 確認值,本次載入值,一致
筆數,6632,6632,是
起始日期,1999-01-05,1999-01-05,是
結束日期,2026-01-20,2026-01-20,是
欄位,"Open, High, Low, Close, Volume, MA, RSI3, RSI5...","Open, High, Low, Close, Volume, MA, RSI3, RSI5...",是


===> [通過] 資料載入一致性
      6,632 筆與階段 3 完全一致；研究期間切出 6,391 個交易日（2000-01-04 ~ 2026-01-20）


---
## 2. 建立部位序列

`position` = 1（持有）／0（空手），直接由 `regime` 對應，**不做任何 shift**。

三項驗證：

1. `position` 的切換日期與階段 3 的 `regime_change == True` 完全一致
2. 研究期間 `position == 1` 的天數等於 `regime == LONG_OK` 的天數（階段 3 為 4,264 天）
3. 目視確認 `position` 在 `regime_change` **當日**就改變，沒有延後一天

In [3]:
df["position"] = (df["regime"] == LONG_OK).astype(int)

# 部位切換日（研究期間第一天視為起點，不算切換）
pos_change = df["position"] != df["position"].shift(1)
pos_change.iloc[0] = False
pos_change_dates = df.index[pos_change]

regime_change_dates = df.index[df["regime_change"]]

same_dates = pos_change_dates.equals(regime_change_dates)
n_long = int((df["position"] == 1).sum())
n_regime_long = int((df["regime"] == LONG_OK).sum())

chk = pd.DataFrame({
    "檢查": ["position 切換日 == regime_change 日",
             f"position==1 天數 == regime==LONG_OK 天數",
             f"與階段 3 記錄的 {EXPECTED_LONG_OK_DAYS} 天一致"],
    "值": [f"{len(pos_change_dates)} vs {len(regime_change_dates)} 天",
           f"{n_long} vs {n_regime_long}",
           f"{n_long}"],
    "結果": ["通過" if same_dates else "★ 異常",
             "通過" if n_long == n_regime_long else "★ 異常",
             "通過" if n_long == EXPECTED_LONG_OK_DAYS else "★ 異常"],
})
display(chk)

pos_ok = (chk["結果"] == "通過").all()
record("部位序列與 regime 一致（無額外 shift）", pos_ok,
       f"{len(pos_change_dates)} 個部位切換日與 regime_change 完全相同；"
       f"position==1 共 {n_long:,} 天，與階段 3 的 {EXPECTED_LONG_OK_DAYS:,} 天一致"
       if pos_ok else "部位序列與 regime 不一致，明細見上表")

,檢查,值,結果
0,position 切換日 == regime_change 日,98 vs 98 天,通過
1,position==1 天數 == regime==LONG_OK 天數,3908 vs 3908,通過
2,與階段 3 記錄的 3908 天一致,3908,通過


===> [通過] 部位序列與 regime 一致（無額外 shift）
      98 個部位切換日與 regime_change 完全相同；position==1 共 3,908 天，與階段 3 的 3,908 天一致


### 2a. 目視確認：position 在 regime_change 當日就改變

印出前 3 個切換點前後各 2 天。要看的是 `regime_change == True` 的那一列，
`position` 已經是新值 —— 若 `position` 要到下一列才改變，代表多做了一次 shift。

In [4]:
for d in pos_change_dates[:3]:
    i = df.index.get_loc(d)
    win = df.iloc[max(i - 2, 0): i + 3][["Open", "Close", "regime",
                                         "regime_change", "position"]].copy()
    win["◀"] = np.where(win.index == d, "◀ 切換日", "")
    win.index = win.index.strftime("%Y-%m-%d")
    print(f"\n=== 切換點 {d:%Y-%m-%d} ===")
    display(win)


=== 切換點 2000-05-02 ===


,Open,Close,regime,regime_change,position,◀
Date,,,,,,
2000-04-27,8587.030273,8541.950195,LONG_OK,False,1,
2000-04-28,8594.480469,8824.360352,LONG_OK,False,1,
2000-05-02,8836.830078,8638.750000,FLAT,True,0,◀ 切換日
2000-05-03,8505.459961,8420.000000,FLAT,False,0,
2000-05-04,8444.070312,8425.379883,FLAT,False,0,



=== 切換點 2001-02-01 ===


,Open,Close,regime,regime_change,position,◀
Date,,,,,,
2001-01-30,5685.189941,5792.500000,FLAT,False,0,
2001-01-31,5853.620117,5936.200195,FLAT,False,0,
2001-02-01,5927.250000,5897.930176,LONG_OK,True,1,◀ 切換日
2001-02-02,5958.779785,6049.259766,LONG_OK,False,1,
2001-02-05,6012.430176,5932.419922,LONG_OK,False,1,



=== 切換點 2001-05-02 ===


,Open,Close,regime,regime_change,position,◀
Date,,,,,,
2001-04-27,5505.609863,5416.669922,LONG_OK,False,1,
2001-04-30,5430.589844,5381.669922,LONG_OK,False,1,
2001-05-02,5469.950195,5304.240234,FLAT,True,0,◀ 切換日
2001-05-03,5287.720215,5405.540039,FLAT,False,0,
2001-05-04,5384.299805,5244.049805,FLAT,False,0,


---
## 3. 建立每日報酬序列 ★ 本階段最關鍵的部分

依規則計算 `ret`，進場日與出場日走開盤價。

實作上把每一天分成四類（進場／持有／出場／空手），逐類套公式，
而不是先算一條收盤對收盤的報酬再去修補特例 —— 後者容易漏掉邊界。

研究期間第一天視為進場日（開盤買進），與 A 組起跑點相同。

In [5]:
prev_pos = df["position"].shift(1).fillna(0).astype(int)   # 首日之前視為空手
pos      = df["position"]

is_entry = (pos == 1) & (prev_pos == 0)
is_hold  = (pos == 1) & (prev_pos == 1)
is_exit  = (pos == 0) & (prev_pos == 1)
is_flat  = (pos == 0) & (prev_pos == 0)

# 理論上不可能同時進出場；若發生代表部位序列有問題
if bool((is_entry & is_exit).any()):
    raise RuntimeError("偵測到同一天既是進場日又是出場日，請停止並人工確認部位序列。")

# 四類必須互斥且涵蓋全部交易日
cover = is_entry.astype(int) + is_hold.astype(int) + is_exit.astype(int) + is_flat.astype(int)
if not bool((cover == 1).all()):
    raise RuntimeError("每日分類未互斥或未涵蓋全部交易日。")

# 前一日收盤：研究期間首日取自 1999 年最後一個交易日
prev_close = df["Close"].shift(1)
prev_close.iloc[0] = prev_close_first

ret = pd.Series(0.0, index=df.index, name="ret")
ret[is_entry] = df.loc[is_entry, "Close"] / df.loc[is_entry, "Open"] - 1
ret[is_hold]  = df.loc[is_hold,  "Close"] / prev_close[is_hold] - 1
ret[is_exit]  = df.loc[is_exit,  "Open"]  / prev_close[is_exit] - 1
ret[is_flat]  = 0.0

df["ret"] = ret

day_types = pd.DataFrame({
    "天數": [int(is_entry.sum()), int(is_hold.sum()), int(is_exit.sum()), int(is_flat.sum())],
}, index=["進場日", "持有中", "出場日", "空手"])
day_types.loc["合計"] = day_types.sum()
day_types["佔比"] = (day_types["天數"] / len(df)).map("{:.2%}".format)
display(day_types)

,天數,佔比
進場日,50,0.78%
持有中,3858,60.37%
出場日,49,0.77%
空手,2434,38.08%
合計,6391,100.00%


### 3a. 前 3 次完整交易的逐日明細

從進場日前一天列到出場日後一天，人工檢查三件事：

- **進場日**的 `ret` 等於 `Close/Open − 1`（不是 `Close/前收 − 1`）
- **出場日**的 `ret` 等於 `Open/前收 − 1`（不是 `Close/前收 − 1`）
- **出場日之後**的 `ret` 為 0

In [6]:
entry_dates = df.index[is_entry]
exit_dates  = df.index[is_exit]

for k in range(3):
    e_in = entry_dates[k]
    later = exit_dates[exit_dates > e_in]
    e_out = later[0] if len(later) else df.index[-1]
    i0, i1 = df.index.get_loc(e_in), df.index.get_loc(e_out)
    win = df.iloc[max(i0 - 1, 0): min(i1 + 2, len(df))][
        ["Open", "Close", "position", "ret"]].copy()
    win["前收"] = prev_close.loc[win.index]
    win["類型"] = np.select(
        [is_entry.loc[win.index], is_hold.loc[win.index],
         is_exit.loc[win.index], is_flat.loc[win.index]],
        ["進場", "持有", "出場", "空手"], default="")
    win["ret 檢算"] = np.select(
        [win["類型"] == "進場", win["類型"] == "持有", win["類型"] == "出場"],
        [win["Close"] / win["Open"] - 1,
         win["Close"] / win["前收"] - 1,
         win["Open"] / win["前收"] - 1], default=0.0)
    win = win[["Open", "Close", "前收", "position", "類型", "ret", "ret 檢算"]]
    win.index = win.index.strftime("%Y-%m-%d")
    print(f"\n=== 第 {k + 1} 筆交易：{e_in:%Y-%m-%d} 進場，{e_out:%Y-%m-%d} 出場"
          f"（僅顯示頭尾，中間持有日省略）===")
    display(win.head(4).style.format({"Open": "{:,.2f}", "Close": "{:,.2f}",
                                      "前收": "{:,.2f}", "ret": "{:+.4%}",
                                      "ret 檢算": "{:+.4%}"}))
    display(win.tail(3).style.format({"Open": "{:,.2f}", "Close": "{:,.2f}",
                                      "前收": "{:,.2f}", "ret": "{:+.4%}",
                                      "ret 檢算": "{:+.4%}"}))


=== 第 1 筆交易：2000-01-04 進場，2000-05-02 出場（僅顯示頭尾，中間持有日省略）===


,Open,Close,前收,position,類型,ret,ret 檢算
Date,,,,,,,
2000-01-04,"8,644.91","8,756.55","8,448.84",1,進場,+1.2914%,+1.2914%
2000-01-05,"8,690.60","8,849.87","8,756.55",1,持有,+1.0657%,+1.0657%
2000-01-06,"8,900.56","8,922.03","8,849.87",1,持有,+0.8154%,+0.8154%
2000-01-07,"8,853.43","8,849.87","8,922.03",1,持有,-0.8088%,-0.8088%


,Open,Close,前收,position,類型,ret,ret 檢算
Date,,,,,,,
2000-04-28,"8,594.48","8,824.36","8,541.95",1,持有,+3.3062%,+3.3062%
2000-05-02,"8,836.83","8,638.75","8,824.36",0,出場,+0.1413%,+0.1413%
2000-05-03,"8,505.46","8,420.00","8,638.75",0,空手,+0.0000%,+0.0000%



=== 第 2 筆交易：2001-02-01 進場，2001-05-02 出場（僅顯示頭尾，中間持有日省略）===


,Open,Close,前收,position,類型,ret,ret 檢算
Date,,,,,,,
2001-01-31,"5,853.62","5,936.20","5,792.50",0,空手,+0.0000%,+0.0000%
2001-02-01,"5,927.25","5,897.93","5,936.20",1,進場,-0.4947%,-0.4947%
2001-02-02,"5,958.78","6,049.26","5,897.93",1,持有,+2.5658%,+2.5658%
2001-02-05,"6,012.43","5,932.42","6,049.26",1,持有,-1.9315%,-1.9315%


,Open,Close,前收,position,類型,ret,ret 檢算
Date,,,,,,,
2001-04-30,"5,430.59","5,381.67","5,416.67",1,持有,-0.6462%,-0.6462%
2001-05-02,"5,469.95","5,304.24","5,381.67",0,出場,+1.6404%,+1.6404%
2001-05-03,"5,287.72","5,405.54","5,304.24",0,空手,+0.0000%,+0.0000%



=== 第 3 筆交易：2001-12-03 進場，2002-05-02 出場（僅顯示頭尾，中間持有日省略）===


,Open,Close,前收,position,類型,ret,ret 檢算
Date,,,,,,,
2001-11-30,"4,497.25","4,441.12","4,465.83",0,空手,+0.0000%,+0.0000%
2001-12-03,"4,534.37","4,646.61","4,441.12",1,進場,+2.4753%,+2.4753%
2001-12-04,"4,638.89","4,766.43","4,646.61",1,持有,+2.5787%,+2.5787%
2001-12-05,"4,892.31","4,924.56","4,766.43",1,持有,+3.3176%,+3.3176%


,Open,Close,前收,position,類型,ret,ret 檢算
Date,,,,,,,
2002-04-30,"6,201.84","6,065.73","6,205.09",1,持有,-2.2459%,-2.2459%
2002-05-02,"6,099.27","5,867.83","6,065.73",0,出場,+0.5529%,+0.5529%
2002-05-03,"5,785.73","5,910.32","5,867.83",0,空手,+0.0000%,+0.0000%


In [7]:
# 空手日報酬必須全為 0（出場日 position 已是 0 但仍有報酬，需排除）
flat_nonzero = int((df.loc[is_flat, "ret"] != 0).sum())

# 極值檢查：對照當日實際行情，確認不是計算錯誤
extreme = df["ret"].abs().nlargest(8)
ext_tbl = pd.DataFrame({
    "ret": df.loc[extreme.index, "ret"],
    "類型": np.select(
        [is_entry.loc[extreme.index], is_hold.loc[extreme.index],
         is_exit.loc[extreme.index]], ["進場", "持有", "出場"], default="空手"),
    "當日 Close/前收 −1": (df.loc[extreme.index, "Close"] /
                           prev_close.loc[extreme.index] - 1),
})
ext_tbl.index = ext_tbl.index.strftime("%Y-%m-%d")
display(ext_tbl.style.format({"ret": "{:+.2%}", "當日 Close/前收 −1": "{:+.2%}"}))

max_abs = df["ret"].abs().max()
ret_ok = (flat_nonzero == 0) and (max_abs <= 0.10)

record("每日報酬序列", ret_ok,
       f"空手日 {int(is_flat.sum()):,} 天報酬全為 0；"
       f"單日報酬最大絕對值 {max_abs:.2%}，未超過 ±10%，"
       f"最大者對應 {ext_tbl.index[0]}（該日實際行情 "
       f"{ext_tbl['當日 Close/前收 −1'].iloc[0]:+.2%}）"
       if ret_ok else
       f"空手日有 {flat_nonzero} 天報酬非 0，或單日報酬 {max_abs:.2%} 超過 ±10%，需人工確認")

,ret,類型,當日 Close/前收 −1
Date,,,
2009-04-30,+6.74%,持有,+6.74%
2004-03-22,-6.68%,持有,-6.68%
2000-03-13,-6.55%,持有,-6.55%
2018-10-11,-6.31%,持有,-6.31%
2001-12-06,+5.77%,持有,+5.77%
2020-01-30,-5.75%,持有,-5.75%
2009-05-04,+5.64%,持有,+5.64%
2004-03-29,+5.57%,持有,+5.57%


===> [通過] 每日報酬序列
      空手日 2,434 天報酬全為 0；單日報酬最大絕對值 6.74%，未超過 ±10%，最大者對應 2009-04-30（該日實際行情 +6.74%）


---
## 4. 交易明細表

C 組每一筆交易，完整列出不截斷。

「期間最大不利變動」（MAE）= 持有期間內收盤價相對**進場價**的最低點跌幅，
用來看單筆交易最難熬的時候有多痛。取樣範圍是進場日到出場日前一天的收盤價
（出場日只有開盤價參與成交，不納入收盤取樣）。

資料結束時仍持有的最後一筆標記為「持有中」，用最後一日收盤價計算。

In [8]:
trades = []
for k, e_in in enumerate(entry_dates, start=1):
    later = exit_dates[exit_dates > e_in]
    open_trade = len(later) == 0
    e_out = later[0] if not open_trade else df.index[-1]

    i0, i1 = df.index.get_loc(e_in), df.index.get_loc(e_out)
    entry_px = df.loc[e_in, "Open"]
    exit_px  = df.loc[e_out, "Close"] if open_trade else df.loc[e_out, "Open"]

    # MAE 取樣：進場日 ~ 出場日前一天的收盤（持有中的最後一天）
    hold_close = df["Close"].iloc[i0: i1] if not open_trade else df["Close"].iloc[i0: i1 + 1]
    mae = hold_close.min() / entry_px - 1

    trades.append({
        "#": k,
        "進場日": f"{e_in:%Y-%m-%d}",
        "進場價（開盤）": entry_px,
        "出場日": f"{e_out:%Y-%m-%d}" + ("（持有中）" if open_trade else ""),
        "出場價": exit_px,
        "持有交易日數": i1 - i0 + (1 if open_trade else 0),
        "報酬率": exit_px / entry_px - 1,
        "期間最大不利變動": mae,
        "持有中": open_trade,
    })

trades = pd.DataFrame(trades).set_index("#")
print(f"C 組共 {len(trades)} 筆交易"
      f"（其中 {int(trades['持有中'].sum())} 筆於資料結束時仍持有）")
display(trades.drop(columns="持有中").style.format({
    "進場價（開盤）": "{:,.2f}", "出場價": "{:,.2f}",
    "報酬率": "{:+.2%}", "期間最大不利變動": "{:.2%}"}))

C 組共 50 筆交易（其中 1 筆於資料結束時仍持有）


,進場日,進場價（開盤）,出場日,出場價,持有交易日數,報酬率,期間最大不利變動
#,,,,,,,
1,2000-01-04,"8,644.91",2000-05-02,"8,836.83",76,+2.22%,-1.26%
2,2001-02-01,"5,927.25",2001-05-02,"5,469.95",61,-7.72%,-9.68%
3,2001-12-03,"4,534.37",2002-05-02,"6,099.27",97,+34.51%,2.48%
4,2002-11-01,"4,596.69",2003-01-02,"4,460.57",43,-2.96%,-3.14%
5,2003-02-06,"4,975.65",2003-03-03,"4,483.44",16,-9.89%,-10.92%
6,2003-06-02,"4,620.54",2003-12-01,"5,768.69",127,+24.85%,1.25%
7,2004-02-02,"6,379.98",2004-04-01,"6,504.54",43,+1.95%,-3.88%
8,2004-09-01,"5,799.82",2004-11-01,"5,725.65",41,-1.28%,-2.57%
9,2005-01-03,"6,166.39",2005-04-01,"6,010.71",57,-2.52%,-6.40%


In [9]:
win_rate = (trades["報酬率"] > 0).mean()
tr_summary = pd.DataFrame({
    "值": [
        f"{len(trades)}",
        f"{int((trades['報酬率'] > 0).sum())}",
        f"{win_rate:.1%}",
        f"{trades['報酬率'].mean():+.2%}",
        f"{trades['報酬率'].median():+.2%}",
        f"{trades['報酬率'].max():+.2%}",
        f"{trades['報酬率'].min():+.2%}",
        f"{trades['持有交易日數'].median():.0f}",
        f"{trades['期間最大不利變動'].min():.2%}",
        f"{trades['期間最大不利變動'].median():.2%}",
    ]
}, index=["交易筆數", "獲利筆數", "勝率", "平均報酬", "中位數報酬",
          "最佳單筆", "最差單筆", "持有交易日數中位數",
          "最深單筆 MAE", "MAE 中位數"])
tr_summary.index.name = "交易統計"
display(tr_summary)

record("交易明細表", True,
       f"C 組研究期間共 {len(trades)} 筆交易，勝率 {win_rate:.1%}，"
       f"平均報酬 {trades['報酬率'].mean():+.2%}，"
       f"最差單筆 {trades['報酬率'].min():+.2%}，"
       f"最深單筆 MAE {trades['期間最大不利變動'].min():.2%}")

,值
交易統計,
交易筆數,50
獲利筆數,26
勝率,52.0%
平均報酬,+4.99%
中位數報酬,+0.53%
最佳單筆,+59.85%
最差單筆,-11.40%
持有交易日數中位數,59
最深單筆 MAE,-14.22%


===> [通過] 交易明細表
      C 組研究期間共 50 筆交易，勝率 52.0%，平均報酬 +4.99%，最差單筆 -11.40%，最深單筆 MAE -14.22%


---
## 5. 權益曲線

A 組與 C 組，起始值 1.0，日報酬累乘。

驗證兩件事：曲線無缺值、無非正值；A 組最終值應約等於
`最後一日收盤 / 首日開盤` —— 這是一個完全獨立於報酬序列的算法，
能抓出累乘過程中的錯誤。

In [10]:
# A 組：首日開盤買進，之後收盤對收盤
ret_A = df["Close"] / prev_close - 1
ret_A.iloc[0] = df["Close"].iloc[0] / df["Open"].iloc[0] - 1
ret_A = ret_A.rename("ret_A")

eq_A = (1 + ret_A).cumprod().rename("equity_A")
eq_C = (1 + df["ret"]).cumprod().rename("equity_C")

expected_A = df["Close"].iloc[-1] / df["Open"].iloc[0]
diff_A = abs(eq_A.iloc[-1] - expected_A)

curve_chk = pd.DataFrame({
    "檢查": ["A 組權益曲線缺值", "C 組權益曲線缺值",
             "A 組權益曲線非正值", "C 組權益曲線非正值",
             "A 組期末權益 vs 末收盤/首開盤"],
    "值": [int(eq_A.isna().sum()), int(eq_C.isna().sum()),
           int((eq_A <= 0).sum()), int((eq_C <= 0).sum()),
           f"{eq_A.iloc[-1]:.6f} vs {expected_A:.6f}（差 {diff_A:.2e}）"],
})
curve_chk["結果"] = ["通過" if v == 0 else "★ 異常" for v in curve_chk["值"][:4]] + \
                    ["通過" if diff_A < 1e-9 else "★ 異常"]
display(curve_chk)

curve_ok = (curve_chk["結果"] == "通過").all()
record("權益曲線", curve_ok,
       f"A、C 兩條曲線皆無缺值、無非正值；A 組期末權益 {eq_A.iloc[-1]:.4f} 與"
       f"「末日收盤/首日開盤」{expected_A:.4f} 差異 {diff_A:.2e}，累乘計算正確"
       if curve_ok else "權益曲線有問題，明細見上表")

,檢查,值,結果
0,A 組權益曲線缺值,0,通過
1,C 組權益曲線缺值,0,通過
2,A 組權益曲線非正值,0,通過
3,C 組權益曲線非正值,0,通過
4,A 組期末權益 vs 末收盤/首開盤,3.673837 vs 3.673837（差 7.11e-15）,通過


===> [通過] 權益曲線
      A、C 兩條曲線皆無缺值、無非正值；A 組期末權益 3.6738 與「末日收盤/首日開盤」3.6738 差異 7.11e-15，累乘計算正確


---
## 6. 績效指標

### 最大回撤：每日 vs 交易層級

**最大回撤一律以每日權益曲線計算。**

有些回測只在交易結束時記錄一次權益，用那條序列算回撤 —— 這會**系統性低估**真實回撤，
因為持有期間內的谷底完全看不到。本專案兩者都算並列呈現，讓這個差距可見：

- `MDD（每日）` —— 每日權益曲線的回撤，這是實際承受的
- `MDD（交易層級）` —— 僅由每筆交易結束時點的權益構成的序列算出的回撤

空手期間權益不變，因此空手不會產生回撤 —— 這正是濾網的作用。

In [11]:
def drawdown(equity):
    """回傳回撤序列（相對歷史高點）。"""
    return equity / equity.cummax() - 1

def dd_detail(equity):
    """最大回撤的幅度、起始（前高）日、谷底日、恢復日。"""
    dd = drawdown(equity)
    trough = dd.idxmin()
    mdd = dd.loc[trough]
    peak = equity.loc[:trough].idxmax()
    after = equity.loc[trough:]
    rec = after[after >= equity.loc[peak]]
    recovered = rec.index[0] if len(rec) else None
    return mdd, peak, trough, recovered

def perf_stats(returns, label, position=None, trade_rets=None, trade_equity=None):
    """由日報酬序列計算績效指標。無風險利率 = 0。"""
    r = returns.dropna()
    eq = (1 + r).cumprod()
    n = len(r)

    total  = eq.iloc[-1] - 1
    cagr   = eq.iloc[-1] ** (TRADING_DAYS_PER_YEAR / n) - 1
    vol    = r.std(ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)
    sharpe = (cagr - RISK_FREE) / vol if vol > 0 else np.nan

    mdd, peak, trough, recovered = dd_detail(eq)
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan

    mdd_trade = np.nan
    if trade_equity is not None and len(trade_equity) > 1:
        mdd_trade = drawdown(trade_equity).min()

    out = {
        "總報酬率": total,
        "年化報酬率": cagr,
        "年化波動度": vol,
        "Sharpe": sharpe,
        "MDD（每日）": mdd,
        "MDD（交易層級）": mdd_trade,
        "MDD 起始日": f"{peak:%Y-%m-%d}",
        "MDD 谷底日": f"{trough:%Y-%m-%d}",
        "MDD 恢復日": f"{recovered:%Y-%m-%d}" if recovered is not None else "尚未恢復",
        "Calmar": calmar,
        "在場時間比例": (position == 1).mean() if position is not None else 1.0,
        "交易次數": len(trade_rets) if trade_rets is not None else 1,
        "勝率": (np.asarray(trade_rets) > 0).mean() if trade_rets is not None and len(trade_rets) else np.nan,
        "交易日數": n,
    }
    return pd.Series(out, name=label)

print("perf_stats 已定義")

perf_stats 已定義


### 交易層級權益序列

由每筆交易的報酬依序累乘而成，只在交易結束時點取值。
與每日權益曲線並列，就能看出「只看交易結束時點」會低估多少回撤。

In [12]:
def trade_equity_curve(trade_returns):
    """由每筆交易報酬構成的權益序列，起點 1.0。"""
    return pd.Series(np.concatenate([[1.0], np.cumprod(1 + np.asarray(trade_returns))]))

teq_all = trade_equity_curve(trades["報酬率"].to_numpy())

cmp_dd = pd.DataFrame({
    "MDD（每日權益）":   [drawdown(eq_C).min()],
    "MDD（交易層級）":   [drawdown(teq_all).min()],
}, index=["C 組（全期間）"])
cmp_dd["低估幅度（百分點）"] = (cmp_dd["MDD（交易層級）"] - cmp_dd["MDD（每日權益）"]) * 100
display(cmp_dd.style.format({"MDD（每日權益）": "{:.2%}", "MDD（交易層級）": "{:.2%}",
                             "低估幅度（百分點）": "{:+.1f}"}))

record("MDD 每日 vs 交易層級", True,
       f"C 組每日權益 MDD {drawdown(eq_C).min():.2%}，"
       f"交易層級 MDD {drawdown(teq_all).min():.2%}，"
       f"交易層級低估 {(drawdown(teq_all).min() - drawdown(eq_C).min()) * 100:+.1f} 個百分點"
       "（僅記錄事實）")

,MDD（每日權益）,MDD（交易層級）,低估幅度（百分點）
C 組（全期間）,-30.38%,-22.21%,+8.2


===> [通過] MDD 每日 vs 交易層級
      C 組每日權益 MDD -30.38%，交易層級 MDD -22.21%，交易層級低估 +8.2 個百分點（僅記錄事實）


---
## 7. 績效比較表

A 組與 C 組，分全期間、IS、OOS 三段。

**IS 與 OOS 的權益曲線各自從 1.0 重新起算**，
所以 IS 的總報酬率不是全期間報酬的一部分，兩者不能相加也不能直接比較 ——
每一段都是獨立的「假設從這一天開始投入 1 元」的結果。

In [13]:
WINDOWS = [
    ("全期間", STUDY_START, f"{df.index[-1]:%Y-%m-%d}"),
    ("IS",     STUDY_START, IS_END),
    ("OOS",    OOS_START,   f"{df.index[-1]:%Y-%m-%d}"),
]

def trades_in(a, b):
    """進場日落在 [a, b] 內的交易。"""
    d = pd.to_datetime(trades["進場日"])
    return trades[(d >= pd.Timestamp(a)) & (d <= pd.Timestamp(b))]

blocks = []
for wname, a, b in WINDOWS:
    rA = ret_A.loc[a:b]
    rC = df["ret"].loc[a:b]
    tr = trades_in(a, b)
    teq = trade_equity_curve(tr["報酬率"].to_numpy())

    sA = perf_stats(rA, "A 買進持有")
    sC = perf_stats(rC, "C 只有濾網", position=df["position"].loc[a:b],
                    trade_rets=tr["報酬率"].to_numpy(), trade_equity=teq)
    blk = pd.concat([sA, sC], axis=1)
    blk.columns = pd.MultiIndex.from_product([[wname], blk.columns])
    blocks.append(blk)

perf = pd.concat(blocks, axis=1)

PCT_ROWS = ["總報酬率", "年化報酬率", "年化波動度", "MDD（每日）",
            "MDD（交易層級）", "在場時間比例", "勝率"]
NUM_ROWS = ["Sharpe", "Calmar"]

def fmt_perf(v, row):
    if pd.isna(v):
        return "—"
    if row in PCT_ROWS:
        return f"{v:.2%}"
    if row in NUM_ROWS:
        return f"{v:.2f}"
    if row in ("交易次數", "交易日數"):
        return f"{int(v):,}"
    return v

perf_disp = perf.copy()
for r in perf_disp.index:
    perf_disp.loc[r] = [fmt_perf(v, r) for v in perf_disp.loc[r]]
display(perf_disp)

全期間                      IS                     OOS            
               A 買進持有      C 只有濾網      A 買進持有      C 只有濾網      A 買進持有      C 只有濾網
總報酬率          267.38%     622.53%      23.11%     226.32%     198.42%     121.42%
年化報酬率           5.11%       7.88%       1.16%       6.75%      14.67%      10.46%
年化波動度          20.52%      13.40%      21.50%      13.69%      18.11%      12.72%
Sharpe           0.25        0.59        0.05        0.49        0.81        0.82
MDD（每日）       -66.22%     -30.38%     -66.22%     -25.73%     -31.63%     -30.38%
MDD（交易層級）           —     -22.21%           —     -16.92%           —     -22.21%
MDD 起始日    2000-02-17  2021-07-15  2000-02-17  2010-01-15  2022-01-04  2021-07-15
MDD 谷底日    2001-10-03  2022-12-29  2001-10-03  2012-07-26  2022-10-25  2022-12-29
MDD 恢復日    2017-06-05  2024-06-14  2017-06-05  2017-03-16  2024-02-15  2024-06-14
Calmar           0.08        0.26        0.02        0.26        0.46        0.34
在場時間比例        100.00%      61.15%     100.00%      59.79%     100.00%      64.23%
交易次數                1          50           1          32           1          18
勝率                  —      52.00%           —      46.88%           —      61.11%
交易日數            6,391       6,391       4,434       4,434       1,957       1,957

In [14]:
record("績效比較表", True,
       f"全期間年化報酬 A {perf[('全期間', 'A 買進持有')]['年化報酬率']:.2%} / "
       f"C {perf[('全期間', 'C 只有濾網')]['年化報酬率']:.2%}；"
       f"MDD（每日）A {perf[('全期間', 'A 買進持有')]['MDD（每日）']:.2%} / "
       f"C {perf[('全期間', 'C 只有濾網')]['MDD（每日）']:.2%}；"
       f"Sharpe A {perf[('全期間', 'A 買進持有')]['Sharpe']:.2f} / "
       f"C {perf[('全期間', 'C 只有濾網')]['Sharpe']:.2f}")

===> [通過] 績效比較表
      全期間年化報酬 A 5.11% / C 7.88%；MDD（每日）A -66.22% / C -30.38%；Sharpe A 0.25 / C 0.59


---
## 8. 逐年報酬

A 組與 C 組逐年對照，並標記 C 組贏或輸。

之後把年份分成「A 組下跌年」與「A 組上漲年」兩組分別統計 ——
濾網的價值理論上集中在下跌年，代價則出現在上漲年。這一格把兩邊的數字都列出來。

2026 年只有 13 個交易日，屬部分年度，會另行標記。

In [15]:
def yearly(r):
    return r.groupby(r.index.year).apply(lambda s: (1 + s).prod() - 1)

yr = pd.DataFrame({"A 買進持有": yearly(ret_A), "C 只有濾網": yearly(df["ret"])})
yr["C − A"] = yr["C 只有濾網"] - yr["A 買進持有"]
yr["C 勝負"] = np.where(yr["C − A"] > 0, "勝", np.where(yr["C − A"] < 0, "負", "平"))
yr["在場比例"] = df["position"].groupby(df.index.year).mean()
yr["交易日數"] = df["ret"].groupby(df.index.year).size()
yr["備註"] = np.where(yr["交易日數"] < 200, "部分年度", "")
yr.index.name = "年份"

display(yr.style.format({"A 買進持有": "{:+.2%}", "C 只有濾網": "{:+.2%}",
                         "C − A": "{:+.2%}", "在場比例": "{:.0%}"}))

,A 買進持有,C 只有濾網,C − A,C 勝負,在場比例,交易日數,備註
年份,,,,,,,
2000,-45.12%,+2.22%,+47.34%,勝,31%,245,
2001,+17.02%,+12.98%,-4.04%,負,33%,245,
2002,-19.79%,+6.42%,+26.22%,勝,48%,248,
2003,+32.30%,+12.70%,-19.60%,負,57%,249,
2004,+4.23%,+0.65%,-3.58%,負,34%,250,
2005,+6.66%,+4.37%,-2.29%,負,58%,247,
2006,+19.48%,+23.85%,+4.37%,勝,73%,247,
2007,+8.72%,+4.37%,-4.35%,負,84%,243,
2008,-46.03%,+5.17%,+51.20%,勝,25%,249,


In [16]:
full = yr[yr["備註"] == ""]          # 排除部分年度
down = full[full["A 買進持有"] < 0]
up   = full[full["A 買進持有"] >= 0]

def grp(g, label):
    return pd.Series({
        "年數": len(g),
        "A 平均報酬": g["A 買進持有"].mean(),
        "C 平均報酬": g["C 只有濾網"].mean(),
        "C − A 平均": g["C − A"].mean(),
        "C 勝出年數": int((g["C − A"] > 0).sum()),
        "C 勝出比例": (g["C − A"] > 0).mean(),
        "C 平均在場比例": g["在場比例"].mean(),
    }, name=label)

grp_tbl = pd.concat([grp(down, "A 下跌年"), grp(up, "A 上漲年"),
                     grp(full, "全部完整年度")], axis=1)
display(grp_tbl.style.format({
    "年數": "{:.0f}", "A 平均報酬": "{:+.2%}", "C 平均報酬": "{:+.2%}",
    "C − A 平均": "{:+.2%}", "C 勝出年數": "{:.0f}",
    "C 勝出比例": "{:.0%}", "C 平均在場比例": "{:.0%}"}))

record("逐年報酬對照", True,
       f"完整年度 {len(full)} 年中 C 組勝出 {int((full['C − A'] > 0).sum())} 年；"
       f"A 下跌的 {len(down)} 年 C 組平均 {down['C 只有濾網'].mean():+.2%} vs "
       f"A {down['A 買進持有'].mean():+.2%}；"
       f"A 上漲的 {len(up)} 年 C 組平均 {up['C 只有濾網'].mean():+.2%} vs "
       f"A {up['A 買進持有'].mean():+.2%}")

,A 下跌年,A 上漲年,全部完整年度
年數,7.000000,19.000000,26.000000
A 平均報酬,-0.247914,0.201024,0.080156
C 平均報酬,-0.054666,0.138561,0.086538
C − A 平均,0.193247,-0.062463,0.006382
C 勝出年數,5.000000,2.000000,7.000000
C 勝出比例,0.714286,0.105263,0.269231
C 平均在場比例,0.415734,0.683892,0.611695


===> [通過] 逐年報酬對照
      完整年度 26 年中 C 組勝出 7 年；A 下跌的 7 年 C 組平均 -5.47% vs A -24.79%；A 上漲的 19 年 C 組平均 +13.86% vs A +20.10%


---
## 9. 視覺化

### 圖 1｜權益曲線（對數座標）

灰色背景為空手（`FLAT`）區間，虛線為 IS/OOS 分界。
對數座標讓 26 年間不同量級的漲跌都能看清楚。

In [17]:
def shade_flat(ax, position):
    """把 position == 0 的連續區間畫成背景色塊。"""
    s = position
    grp_id = (s != s.shift(1)).cumsum()
    for _, seg in s.groupby(grp_id):
        if seg.iloc[0] == 0:
            i = df.index.get_loc(seg.index[-1])
            end = df.index[min(i + 1, len(df) - 1)]
            ax.axvspan(seg.index[0], end, color="grey", alpha=0.20, lw=0)

fig, ax = plt.subplots(figsize=(15, 7))
ax.plot(eq_A.index, eq_A, lw=1.0, color="#8a8a8a", label="A: Buy & Hold")
ax.plot(eq_C.index, eq_C, lw=1.3, color="#3b6ea5", label="C: MA50 monthly filter only")
shade_flat(ax, df["position"])
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=1.2)
ax.text(pd.Timestamp(OOS_START), eq_A.max(), "  IS | OOS", va="top", ha="left", fontsize=10)
ax.set_yscale("log")
ax.set_title("Equity curves (start = 1.0, log scale) — shaded = out of market")
ax.set_xlabel("Date"); ax.set_ylabel("Equity (log)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_7152\1170444545.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 2｜回撤曲線（每日權益）

In [18]:
dd_A, dd_C = drawdown(eq_A), drawdown(eq_C)

fig, ax = plt.subplots(figsize=(15, 6))
ax.fill_between(dd_A.index, dd_A * 100, 0, color="#8a8a8a", alpha=0.55, label="A: Buy & Hold")
ax.fill_between(dd_C.index, dd_C * 100, 0, color="#3b6ea5", alpha=0.60, label="C: Filter only")
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=1.2)
ax.set_title("Drawdown from daily equity curve")
ax.set_xlabel("Date"); ax.set_ylabel("Drawdown (%)")
ax.legend(loc="lower left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_7152\3277885610.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 3｜逐年報酬對照

In [19]:
x = np.arange(len(yr))
w = 0.4
fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - w / 2, yr["A 買進持有"] * 100, w, color="#8a8a8a", label="A: Buy & Hold")
ax.bar(x + w / 2, yr["C 只有濾網"] * 100, w, color="#3b6ea5", label="C: Filter only")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(yr.index, rotation=45)
ax.set_title("Annual returns (2026 is a partial year)")
ax.set_xlabel("Year"); ax.set_ylabel("Return (%)")
ax.legend(); ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

C:\Users\king5\AppData\Local\Temp\ipykernel_7152\2379013936.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### 圖 4｜滾動 1 年報酬差異（C − A）

滾動 245 個交易日的累積報酬差。零線以上代表過去一年 C 組領先 A 組。
用來看超額報酬集中在哪些時期 —— 是均勻累積，還是靠少數幾段大跌拉開。

In [20]:
W = TRADING_DAYS_PER_YEAR
roll_A = eq_A / eq_A.shift(W) - 1
roll_C = eq_C / eq_C.shift(W) - 1
roll_diff = (roll_C - roll_A).dropna()

fig, ax = plt.subplots(figsize=(15, 6))
ax.fill_between(roll_diff.index, roll_diff * 100, 0,
                where=(roll_diff >= 0), color="#2e7d32", alpha=0.6, label="C outperforms")
ax.fill_between(roll_diff.index, roll_diff * 100, 0,
                where=(roll_diff < 0), color="#b3261e", alpha=0.6, label="C underperforms")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(pd.Timestamp(OOS_START), color="black", ls="--", lw=1.2)
ax.set_title(f"Rolling {W}-day return difference (C − A)")
ax.set_xlabel("Date"); ax.set_ylabel("Difference (percentage points)")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

roll_tbl = pd.DataFrame({"值": [
    f"{(roll_diff > 0).mean():.1%}",
    f"{roll_diff.max():+.1%}  ({roll_diff.idxmax():%Y-%m-%d})",
    f"{roll_diff.min():+.1%}  ({roll_diff.idxmin():%Y-%m-%d})",
]}, index=["C 領先的時間比例", "最大領先", "最大落後"])
roll_tbl.index.name = f"滾動 {W} 交易日報酬差"
display(roll_tbl)

C:\Users\king5\AppData\Local\Temp\ipykernel_7152\1494195291.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


,值
滾動 245 交易日報酬差,
C 領先的時間比例,40.9%
最大領先,+59.3% (2008-11-24)
最大落後,-49.8% (2021-03-22)


---
## 10. 存檔

`data/backtest_C.csv` —— 日期索引 + position、ret、equity_C、equity_A、drawdown_C、drawdown_A
`data/trades_C.csv` —— 交易明細

存檔後讀回驗證一致性。

In [21]:
bt = pd.DataFrame({
    "position":    df["position"],
    "ret":         df["ret"],
    "ret_A":       ret_A,
    "equity_C":    eq_C,
    "equity_A":    eq_A,
    "drawdown_C":  dd_C,
    "drawdown_A":  dd_A,
})
bt.index.name = "Date"
bt.to_csv(DATA_OUT, date_format="%Y-%m-%d")

trades_out = trades.drop(columns="持有中")
trades_out.to_csv(TRADES_OUT, encoding="utf-8-sig")

back_bt = pd.read_csv(DATA_OUT, index_col="Date", parse_dates=True)
back_tr = pd.read_csv(TRADES_OUT, index_col="#")

bt_ok = (len(back_bt) == len(bt) and list(back_bt.columns) == list(bt.columns)
         and np.allclose(back_bt.to_numpy(), bt.to_numpy(), equal_nan=True))
tr_ok = (len(back_tr) == len(trades_out)
         and np.allclose(back_tr["報酬率"].to_numpy(),
                         trades_out["報酬率"].to_numpy(), equal_nan=True))

info = pd.DataFrame({"值": [
    DATA_OUT, f"{os.path.getsize(DATA_OUT):,} bytes", f"{len(back_bt):,}",
    ", ".join(back_bt.columns),
    f"{back_bt.index.min():%Y-%m-%d} ~ {back_bt.index.max():%Y-%m-%d}",
    TRADES_OUT, f"{os.path.getsize(TRADES_OUT):,} bytes", f"{len(back_tr)}",
]}, index=["回測檔", "  └ 大小", "  └ 筆數", "  └ 欄位", "  └ 期間",
           "交易明細檔", "  └ 大小", "  └ 交易筆數"])
info.index.name = "項目"
display(info)

record("存檔與讀回一致性", bt_ok and tr_ok,
       f"{DATA_OUT}（{len(back_bt):,} 筆 × {len(back_bt.columns)} 欄）與 "
       f"{TRADES_OUT}（{len(back_tr)} 筆）皆已寫入，讀回數值完全一致"
       if bt_ok and tr_ok else "讀回的資料與記憶體中不一致，請停止並人工確認")

,值
項目,
回測檔,data/backtest_C.csv
└ 大小,"820,173 bytes"
└ 筆數,"6,391"
└ 欄位,"position, ret, ret_A, equity_C, equity_A, draw..."
└ 期間,2000-01-04 ~ 2026-01-20
交易明細檔,data/trades_C.csv
└ 大小,"5,218 bytes"
└ 交易筆數,50


===> [通過] 存檔與讀回一致性
      data/backtest_C.csv（6,391 筆 × 7 欄）與 data/trades_C.csv（50 筆）皆已寫入，讀回數值完全一致


---
## 11. 小結

In [22]:
summary = pd.DataFrame(CHECKS)
display(summary)

n_fail = int((summary["結果"] == "異常").sum())
print(f"\n共 {len(summary)} 項驗證，通過 {len(summary) - n_fail} 項，異常 {n_fail} 項")

,驗證項目,結果,說明
0,資料載入一致性,通過,"6,632 筆與階段 3 完全一致；研究期間切出 6,391 個交易日（2000-01-04..."
1,部位序列與 regime 一致（無額外 shift）,通過,"98 個部位切換日與 regime_change 完全相同；position==1 共 3,..."
2,每日報酬序列,通過,"空手日 2,434 天報酬全為 0；單日報酬最大絕對值 6.74%，未超過 ±10%，最大者..."
3,交易明細表,通過,C 組研究期間共 50 筆交易，勝率 52.0%，平均報酬 +4.99%，最差單筆 -11....
4,權益曲線,通過,A、C 兩條曲線皆無缺值、無非正值；A 組期末權益 3.6738 與「末日收盤/首日開盤」3...
5,MDD 每日 vs 交易層級,通過,C 組每日權益 MDD -30.38%，交易層級 MDD -22.21%，交易層級低估 +8...
6,績效比較表,通過,全期間年化報酬 A 5.11% / C 7.88%；MDD（每日）A -66.22% / C...
7,逐年報酬對照,通過,完整年度 26 年中 C 組勝出 7 年；A 下跌的 7 年 C 組平均 -5.47% vs...
8,存檔與讀回一致性,通過,"data/backtest_C.csv（6,391 筆 × 7 欄）與 data/trade..."



共 9 項驗證，通過 9 項，異常 0 項


In [23]:
# 小結會引用到的數字集中呈現
def g(w, c, k):
    return perf[(w, c)][k]

key = pd.DataFrame({
    "A 買進持有": [f"{g(w, 'A 買進持有', k):.2%}" if k not in ("Sharpe", "Calmar")
                   else f"{g(w, 'A 買進持有', k):.2f}"
                   for w in ["全期間", "IS", "OOS"]
                   for k in ["年化報酬率", "Sharpe", "MDD（每日）"]],
    "C 只有濾網": [f"{g(w, 'C 只有濾網', k):.2%}" if k not in ("Sharpe", "Calmar")
                   else f"{g(w, 'C 只有濾網', k):.2f}"
                   for w in ["全期間", "IS", "OOS"]
                   for k in ["年化報酬率", "Sharpe", "MDD（每日）"]],
}, index=pd.MultiIndex.from_product([["全期間", "IS", "OOS"],
                                      ["年化報酬率", "Sharpe", "MDD（每日）"]]))
display(key)

extra = pd.DataFrame({"值": [
    f"{len(trades)}",
    f"{(trades['報酬率'] > 0).mean():.1%}",
    f"{df['position'].mean():.2%}",
    f"{drawdown(eq_C).min():.2%} vs {drawdown(teq_all).min():.2%}",
    f"{down['C 只有濾網'].mean():+.2%} vs {down['A 買進持有'].mean():+.2%}（{len(down)} 年）",
    f"{up['C 只有濾網'].mean():+.2%} vs {up['A 買進持有'].mean():+.2%}（{len(up)} 年）",
]}, index=["交易筆數", "勝率", "在場時間比例",
           "MDD 每日 vs 交易層級", "A 下跌年 C vs A 平均", "A 上漲年 C vs A 平均"])
extra.index.name = "其他關鍵數字"
display(extra)

A 買進持有   C 只有濾網
全期間 年化報酬率      5.11%    7.88%
    Sharpe      0.25     0.59
    MDD（每日）  -66.22%  -30.38%
IS  年化報酬率      1.16%    6.75%
    Sharpe      0.05     0.49
    MDD（每日）  -66.22%  -25.73%
OOS 年化報酬率     14.67%   10.46%
    Sharpe      0.81     0.82
    MDD（每日）  -31.63%  -30.38%

,值
其他關鍵數字,
交易筆數,50
勝率,52.0%
在場時間比例,61.15%
MDD 每日 vs 交易層級,-30.38% vs -22.21%
A 下跌年 C vs A 平均,-5.47% vs -24.79%（7 年）
A 上漲年 C vs A 平均,+13.86% vs +20.10%（19 年）


### 本階段結論

以下只陳述數字與觀察到的事實，不評價策略好壞、不建議改進方向、不預測 B 組結果。

**規則實作已驗證**

- `position` 直接由 `regime` 對應，未做任何額外 shift：部位切換日與階段 3 的
  `regime_change == True` 完全相同，`position == 1` 的天數與階段 3 記錄的 4,264 天一致。
- 每日報酬按進場／持有／出場／空手四類分別套公式，四類互斥且涵蓋全部交易日；
  空手日報酬全為 0。前 3 筆交易的逐日明細已列出，進出場日的開盤成交公式套用正確。
- A 組期末權益與「末日收盤 ÷ 首日開盤」的獨立算法一致到 1e-9 以內，累乘計算無誤。

**績效對照**

三個期間（全期間、IS、OOS）的年化報酬、Sharpe、每日 MDD 見上方彙整表，
完整指標見區塊 7。**IS 與 OOS 的權益曲線各自從 1.0 重新起算**，
兩段的總報酬率不能相加，也不能與全期間直接比較。

**最大回撤的兩種算法**

`MDD（每日）` 與 `MDD（交易層級）` 並列於區塊 6 與區塊 7。
交易層級的算法只在交易結束時點取樣，看不到持有期間內的谷底，
因此系統性低估實際承受的回撤，兩者的差距已量化列出。

**濾網在下跌年與上漲年的表現**

區塊 8 把完整年度分成「A 組下跌年」與「A 組上漲年」兩組，
分別列出 A 與 C 的平均報酬、C 勝出的年數與比例、以及 C 的平均在場比例。
區塊 9 圖 4 的滾動 245 日報酬差則顯示超額報酬在時間上的分布。

**交易特徵**

交易筆數、勝率、平均與中位數報酬、持有交易日數中位數、
以及每筆交易的期間最大不利變動（MAE），見區塊 4。

---

### 產出

- `data/backtest_C.csv` —— position、ret、ret_A、equity_C、equity_A、drawdown_C、drawdown_A
- `data/trades_C.csv` —— 每筆交易明細

### 下一階段

本階段完全未使用 RSI。第二層的 RSI 回檔進場訊號屬於階段 5，
B 組回測屬於階段 6。請先人工確認上述績效數字與交易明細，再進行下一階段。